# Smallest Numbers (lower-bound binary search)

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Arrays, Binary Search · **Difficulty/Frequency:** Uncommon (3/10)

## Concepts

**What this problem is really testing:**
- **Lower-bound binary search** — the variant that finds a *boundary*, not an exact match
- The half-open interval `[left, right)` convention, and why it makes the edge cases vanish
- Whether you **notice the ambiguity** in the question and ask about it

**First-principles primer — what is each piece?**

- **Binary search for a boundary, not a value.** Classic binary search asks "where is `x`?" and fails when `x` is absent. **Lower bound** asks a question that *always* has an answer: *"where would `x` go, to keep the array sorted?"* — equivalently, the first index whose value is `>= x`. That is `bisect_left`.
- **The monotone predicate.** Binary search works whenever some yes/no test is `False, False, ..., False, True, True, ..., True` along the array. Here the test is `arr[i] >= target`, and sortedness guarantees exactly that shape. You are binary-searching for **the first True**.
- **The half-open interval `[left, right)`.** `right` starts at `len(arr)`, not `len(arr) - 1`, and means "one past the last candidate". This is the choice that makes the "everything is smaller than target" case fall out for free: the loop simply ends with `left == len(arr)`, no special case needed.

**The invariant — say this before writing any code:**

> Everything strictly left of `left` is `< target`; everything at or right of `right` is `>= target`.

Both halves are trivially true at the start (both regions are empty). Each step preserves them. When `left == right` the unknown region is empty, so `left` **is** the boundary. That is a complete correctness proof in three lines, and it is what turns "I think this is right" into "here is why".

**The ambiguity you should raise.** *"Find all **the smallest numbers** >= the integer"*:

| Reading | `[1, 3, 3, 5, 7]`, target 2 | Size of the answer |
|---|---|---|
| **A** — every number `>= target` (the whole suffix) | indices 1, 2, 3, 4 | O(n) |
| **B** — the smallest such value, and **all its duplicates** | indices 1, 2 (both are 3) | O(k) |

The official answer implements A — but under A the word *"smallest"* does no work at all, which is a strong hint that B was meant. B is also the better problem: it needs **two** boundaries (`bisect_left` and `bisect_right`) and returns a run rather than a tail. The notebook implements both.

**Simple worked example.** `arr = [1, 3, 5, 7, 9]`, `target = 4`:

| step | left | right | mid | `arr[mid]` | action |
|---|---|---|---|---|---|
| 0 | 0 | 5 | 2 | 5 | `5 >= 4` → boundary is here or left → `right = 2` |
| 1 | 0 | 2 | 1 | 3 | `3 < 4` → boundary is further right → `left = 2` |
| 2 | 2 | 2 | — | — | `left == right` → **stop** |

Answer: index **2** (value 5). Three steps for five elements; a scan would have taken three too — but at a million elements it is 20 steps versus a million.

## Problem Statement

Given a **sorted** array and a target integer, return the indices of the smallest numbers `>= target`.

**Reading A** (the official one) — every index whose value is `>= target`:

```python
smallest_numbers([1, 3, 3, 5, 7], 2)   # -> [1, 2, 3, 4]
```

**Reading B** — the smallest qualifying *value*, and every index holding it:

```python
smallest_equal_run([1, 3, 3, 5, 7], 2)  # -> [1, 2]   (both are 3)
```

### Approach 1 — Naive (linear scan)

**Idea:** walk the array and collect every index whose value passes the test.

Correct, and it completely ignores the one piece of structure you were handed. Worth writing as a baseline and as the reference implementation the binary-search versions get tested against — but say out loud that sortedness is being wasted.

**Time complexity:** **O(n)** — every element inspected.

**Space complexity:** O(k) for the output.

In [ ]:
import bisect
from typing import List


def smallest_numbers_linear(arr: List[int], target: int) -> List[int]:
    return [i for i, v in enumerate(arr) if v >= target]      # O(n): ignores sortedness

### Approach 2 — Optimal (lower-bound binary search)

**Idea:** find the **boundary** in O(log n), then take everything from there.

Three details worth defending:

- **`right = len(arr)`, not `len(arr) - 1`.** The half-open interval is what makes "no element qualifies" work with no special case: the loop just ends at `left == len(arr)` and the slice is empty.
- **`left = mid + 1` but `right = mid`.** Asymmetric, and deliberately so. When `arr[mid] < target`, `mid` is definitively *not* the answer, so skip past it. When `arr[mid] >= target`, `mid` **might** be the answer, so it stays inside the search window. Writing `right = mid - 1` here silently discards the correct answer.
- **`mid = (left + right) // 2`** cannot overflow in Python (arbitrary-precision ints), but in C or Java it can for large arrays — the standard fix is `left + (right - left) // 2`. Worth mentioning; it was a real bug in Java's own binary search for nearly a decade.

**Time complexity:** **O(log n)** to find the boundary, plus O(k) to materialise the indices.

**Space complexity:** O(k) for the output, O(1) for the search itself.

In [ ]:
def lower_bound(arr: List[int], target: int) -> int:
    """First index i with arr[i] >= target; len(arr) if there is none. (== bisect_left)"""
    left, right = 0, len(arr)              # half-open [left, right): `right` is ONE PAST the end
    while left < right:
        mid = (left + right) // 2          # in C/Java: left + (right - left) // 2, to avoid overflow
        if arr[mid] < target:
            left = mid + 1                 # mid is definitively too small - skip PAST it
        else:
            right = mid                    # mid might BE the answer - keep it in the window
    return left                            # INVARIANT: arr[:left] < target <= arr[left:]


def smallest_numbers(arr: List[int], target: int) -> List[int]:
    """Reading A: every index whose value is >= target."""
    return list(range(lower_bound(arr, target), len(arr)))

### Approach 3 — Reading B (the smallest qualifying value, and all its duplicates)

**Idea:** the phrase *"all the **smallest** numbers"* most naturally means: find the smallest value that is `>= target`, then return **every** index holding that value.

That needs **two** boundaries:

- `bisect_left(arr, v)` — the first index equal to `v`
- `bisect_right(arr, v)` — one past the last index equal to `v`

The value `v` is simply `arr[lower_bound(arr, target)]` — the first qualifying element. Since the array is sorted, all its copies are **contiguous**, so the two boundaries delimit a run.

Note the first boundary is already known from `lower_bound`, so this costs one extra binary search, not two.

**Time complexity:** **O(log n)** plus O(k) for the k duplicates — and here k is a duplicate count, typically tiny, rather than the whole tail.

**Space complexity:** O(k).

In [ ]:
def upper_bound(arr: List[int], target: int) -> int:
    """One past the last index equal to target. (== bisect_right)"""
    left, right = 0, len(arr)
    while left < right:
        mid = (left + right) // 2
        if arr[mid] <= target:             # <= instead of < : the ONLY difference from lower_bound
            left = mid + 1
        else:
            right = mid
    return left


def smallest_equal_run(arr: List[int], target: int) -> List[int]:
    """Reading B: the smallest value >= target, and every index holding it."""
    start = lower_bound(arr, target)
    if start == len(arr):
        return []                          # nothing qualifies
    value = arr[start]                     # THE smallest qualifying value
    end = upper_bound(arr, value)          # duplicates are contiguous in a sorted array
    return list(range(start, end))

### Approach 4 — `bisect`, the version you would actually ship

**Idea:** Python's standard library already has both boundaries, implemented in C.

Knowing that `bisect_left` **is** the lower bound is the practical half of this question. In an interview, write the loop to show the mechanics, then say you would ship this — that ordering shows both capability and judgement.

`bisect` also takes `lo`/`hi` arguments, which lets you search a sub-range without slicing (and slicing would cost O(n), quietly destroying the O(log n) you came for).

**Time complexity:** O(log n).

**Space complexity:** O(1) for the search.

In [ ]:
def smallest_numbers_bisect(arr: List[int], target: int) -> List[int]:
    return list(range(bisect.bisect_left(arr, target), len(arr)))


def smallest_equal_run_bisect(arr: List[int], target: int) -> List[int]:
    start = bisect.bisect_left(arr, target)
    if start == len(arr):
        return []
    return list(range(start, bisect.bisect_right(arr, arr[start])))

## Verification

Every boundary case: target below everything, above everything, exactly on an element, between elements, and duplicates — plus an exhaustive check against the linear reference on every array/target combination in a range.

In [ ]:
import random

# --- The worked example ---
assert lower_bound([1, 3, 5, 7, 9], 4) == 2
assert smallest_numbers([1, 3, 5, 7, 9], 4) == [2, 3, 4]

# --- lower_bound against every interesting position ---
arr = [1, 3, 3, 5, 7]
assert lower_bound(arr, 0) == 0, "below everything -> index 0"
assert lower_bound(arr, 1) == 0, "exactly the first element"
assert lower_bound(arr, 2) == 1, "between elements"
assert lower_bound(arr, 3) == 1, "on a DUPLICATE -> the FIRST of them"
assert lower_bound(arr, 4) == 3
assert lower_bound(arr, 7) == 4, "exactly the last element"
assert lower_bound(arr, 8) == 5, "above everything -> len(arr), NOT an error"
assert lower_bound([], 5) == 0, "empty array"

# upper_bound differs from lower_bound only on duplicates
assert upper_bound(arr, 3) == 3, "one PAST the last 3"
assert upper_bound(arr, 2) == 1, "no 2 present -> same as lower_bound"
assert upper_bound(arr, 7) == 5
assert upper_bound(arr, 0) == 0

# --- Reading A: the whole suffix ---
for fn in (smallest_numbers, smallest_numbers_bisect, smallest_numbers_linear):
    assert fn([1, 3, 3, 5, 7], 2) == [1, 2, 3, 4], fn.__name__
    assert fn([1, 3, 3, 5, 7], 0) == [0, 1, 2, 3, 4], f"{fn.__name__}: everything qualifies"
    assert fn([1, 3, 3, 5, 7], 99) == [], f"{fn.__name__}: nothing qualifies"
    assert fn([], 5) == [], f"{fn.__name__}: empty array"
    assert fn([5], 5) == [0], f"{fn.__name__}: single element, exact match"
    assert fn([5], 6) == [], fn.__name__
    assert fn([5], 4) == [0], fn.__name__

# --- Reading B: the smallest qualifying value and its duplicates ---
for fn in (smallest_equal_run, smallest_equal_run_bisect):
    assert fn([1, 3, 3, 5, 7], 2) == [1, 2], f"{fn.__name__}: both 3s, not the whole tail"
    assert fn([1, 3, 3, 5, 7], 3) == [1, 2], f"{fn.__name__}: exact match on the duplicates"
    assert fn([1, 3, 3, 5, 7], 4) == [3], f"{fn.__name__}: a single 5"
    assert fn([1, 3, 3, 5, 7], 99) == [], f"{fn.__name__}: nothing qualifies"
    assert fn([], 5) == [], fn.__name__
    assert fn([2, 2, 2, 2], 1) == [0, 1, 2, 3], f"{fn.__name__}: all identical"
    assert fn([2, 2, 2, 2], 3) == [], fn.__name__

# The two readings genuinely differ - this is why you must ask
a = smallest_numbers([1, 3, 3, 5, 7], 2)
b = smallest_equal_run([1, 3, 3, 5, 7], 2)
assert a != b and a == [1, 2, 3, 4] and b == [1, 2], (a, b)

# --- Negative numbers and duplicates at the edges ---
neg = [-10, -5, -5, 0, 3]
assert lower_bound(neg, -7) == 1
assert smallest_numbers(neg, -5) == [1, 2, 3, 4]
assert smallest_equal_run(neg, -7) == [1, 2], "the two -5s"
assert smallest_numbers(neg, -100) == [0, 1, 2, 3, 4]

dupes_at_start = [4, 4, 4, 9]
assert smallest_equal_run(dupes_at_start, 0) == [0, 1, 2]
dupes_at_end = [1, 9, 9, 9]
assert smallest_equal_run(dupes_at_end, 5) == [1, 2, 3]

# --- Exhaustive: every array up to length 6 over a small alphabet, every target ---
random.seed(73)
for _ in range(3000):
    n = random.randint(0, 6)
    a = sorted(random.choices(range(-3, 6), k=n))
    for target in range(-5, 8):
        expected = smallest_numbers_linear(a, target)
        assert smallest_numbers(a, target) == expected, (a, target)
        assert smallest_numbers_bisect(a, target) == expected, (a, target)
        # lower_bound must agree with the standard library, always
        assert lower_bound(a, target) == bisect.bisect_left(a, target), (a, target)
        assert upper_bound(a, target) == bisect.bisect_right(a, target), (a, target)

        # Reading B, checked against an independent brute-force definition
        qualifying = [v for v in a if v >= target]
        if qualifying:
            smallest = min(qualifying)
            ref = [i for i, v in enumerate(a) if v == smallest]
        else:
            ref = []
        assert smallest_equal_run(a, target) == ref, (a, target, ref)
        assert smallest_equal_run_bisect(a, target) == ref, (a, target)

# --- Large arrays: the boundary must still be exact ---
big = list(range(0, 2_000_000, 2))          # every even number
assert lower_bound(big, 1_000_000) == 500_000
assert lower_bound(big, 999_999) == 500_000, "an odd target lands between elements"
assert lower_bound(big, -1) == 0
assert lower_bound(big, 10 ** 9) == len(big)

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Returning values instead of indices.** `arr[lower_bound(arr, target):]` — same O(log n) to find the boundary, same O(k) to build the result. The complexity is unchanged; what changes is that a *slice* copies, so if the caller only wants to iterate, returning the **index** (or a range object) avoids the copy entirely. That is why the question asks for indices.
- **Duplicates of the target.** `lower_bound` returns the **first** occurrence, by construction — because when `arr[mid] == target` it moves `right = mid` rather than stopping. This is precisely why the "found it, return early" optimisation is wrong for a boundary search: stopping at the first equal element you happen to land on gives you *an* occurrence, not the *first* one.
- **Upper bound.** Implemented above, and the diff is a single character: `<` becomes `<=`. That is worth internalising — `bisect_left` and `bisect_right` are the same algorithm with one comparison flipped, and `upper - lower` gives the count of occurrences in O(log n) without ever touching the elements between them.
- **Many queries against the same array.** Each is already O(log n), so there is little to precompute — but if the *value range* is small and dense, an offset table (`answer[v] = lower_bound(arr, v)`) makes each query O(1) after O(range) setup. That is a **counting-sort-shaped** trade: brilliant when values are bounded, useless when they are not. If the queries all arrive at once, a better move is to **sort the queries** and sweep both arrays with two pointers: O(q log q + n) total, beating O(q log n) when q is large.
- **A rotated sorted array.** The monotone predicate is broken — `arr[i] >= target` is no longer `False...False,True...True`, so plain lower bound is invalid. You binary-search for the **pivot** first (the one place where `arr[i] > arr[i+1]`), which is itself a monotone question, then run the ordinary search on whichever half can contain the target. Still O(log n), but the important part is being able to say *why* the original approach fails rather than just patching it.
- **Overflow in `mid`.** `(left + right) // 2` is safe in Python because integers are arbitrary-precision. In C or Java with a large array it overflows to a negative index — a bug that sat in the JDK's own `binarySearch` for nine years. The fix is `left + (right - left) // 2`. Worth a sentence; it signals you have written this in a language without bignums.

## Empirical complexity check

Compare the **linear scan** with the **binary search**, using a target near the **end** of the array so that the result set stays tiny and only the *search* cost varies.

| Growth when the array doubles | What it means |
|---|---|
| ~2x | linear — every element is inspected |
| ~1x | logarithmic — one extra halving step per doubling |

If the target were near the *start*, both would be O(n) simply because the **output** is O(n) — a reminder that `O(log n + k)` has a `k` in it for a reason.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

QUERIES = 2000


def make_array(n):
    arr = list(range(n))
    # Targets near the END, so the RESULT is tiny and only the search cost varies.
    targets = [n - 1 - (i % 5) for i in range(QUERIES)]
    return (arr, targets)


def run_linear(arr, targets):
    for t in targets:
        smallest_numbers_linear(arr, t)         # O(n) every query


def run_binary(arr, targets):
    for t in targets:
        smallest_numbers(arr, t)                # O(log n) + O(k), k <= 5 here


def run_bisect(arr, targets):
    for t in targets:
        smallest_numbers_bisect(arr, t)         # the same, in C


benchmark(
    {"Approach 1 - linear scan O(n)": run_linear,
     "Approach 2 - binary search O(log n)": run_binary,
     "Approach 4 - bisect (C) O(log n)": run_bisect},
    make_array,
    sizes=[2000, 4000, 8000, 16000],
    repeats=2,
)

## Patterns learned

- **Binary search finds boundaries, not just values.** "Where is x?" fails when x is absent; "where would x go?" always has an answer. That reframing is what makes `lower_bound` the workhorse and exact-match search the special case.
- **Look for the monotone predicate.** Binary search applies whenever some test reads `False...False, True...True` along the array. Naming that predicate tells you immediately whether the technique is valid — and, for a rotated array, exactly why it is not.
- **Use the half-open interval `[left, right)`.** `right = len(arr)` makes "nothing qualifies" fall out as `left == len(arr)` with no special case. Almost every off-by-one in binary search traces back to mixing conventions.
- **The asymmetry is deliberate.** `left = mid + 1` (mid is ruled out) but `right = mid` (mid is still a candidate). Making them symmetric discards the answer.
- **State the invariant before you write the loop.** *"Everything left of `left` is smaller; everything from `right` on is not."* Three lines, and the correctness proof is finished before the code exists.
- **`bisect_left` and `bisect_right` differ by one character.** `<` versus `<=`. Their difference is the count of duplicates, obtainable in O(log n) without looking at a single element between them.
- **Ask when the wording is ambiguous.** "All the smallest numbers" has two defensible readings that return different answers. Guessing costs you the whole question; asking costs one sentence.